#### **this notebook is designed to be read after notebooks related to data collection, data modelling, and data querying**

### this sector aims to give the top 10 recommanded crops for each city based on the data
(does constitude any advice to real life agriculture, and may deviate greatly from real condition due to use of synthetic data and other limitations)

#### reccomand crops based on temperature, rain and pollution, assign weight of 0.5, 0.3, 0.2 on them respectively to reflect the importance of these factors

#### load temperature, rain and pollution data of each city as:
- `temprature`: lists of maximum temperature and minimum temperature of each day
- `rainfall`: average rainfall per day
- `pollution`: average quantity of each pollutant of the year


In [13]:
import pandas as pd
from pymongo import MongoClient
import os

# Connect to MongoDB
connection_string = open(os.path.join("..", "mongo_db_connection_string.txt"), "r").read()
client = MongoClient(connection_string)
db = client["WeatherDB"]

pipeline = [
    # Group by city and date to compute daily statistics
    {
        "$group": {
            "_id": {
                "city": "$Location",
                "date": {"$dateToString": {"format": "%Y-%m-%d", "date": "$Timestamp"}}
            },
            "max_temp": {"$max": {"$subtract": ["$Temperature.Temp_Max", 273.15]}},  # Convert to Celsius
            "min_temp": {"$min": {"$subtract": ["$Temperature.Temp_Min", 273.15]}},  # Convert to Celsius
            "total_rainfall": {"$sum": "$Rain.Rain_1h_mm"},
            "avg_AQI": {"$avg": "$Air_Pollution.AQI"},
            "avg_CO": {"$avg": "$Air_Pollution.CO"},
            "avg_NO": {"$avg": "$Air_Pollution.NO"},
            "avg_NO2": {"$avg": "$Air_Pollution.NO2"},
            "avg_O3": {"$avg": "$Air_Pollution.O3"},
            "avg_SO2": {"$avg": "$Air_Pollution.SO2"},
            "avg_PM2_5": {"$avg": "$Air_Pollution.PM2.5"},
            "avg_PM10": {"$avg": "$Air_Pollution.PM10"},
            "avg_NH3": {"$avg": "$Air_Pollution.NH3"}
        }
    },
    # Group by city to compute city-wide statistics
    {
        "$group": {
            "_id": "$_id.city",
            "max_temperature_days": {"$push": "$max_temp"},
            "min_temperature_days": {"$push": "$min_temp"},
            "average_rainfall_per_day": {"$avg": "$total_rainfall"},
            "avg_pollution": {
                "$push": {
                    "AQI": "$avg_AQI",
                    "CO": "$avg_CO",
                    "NO": "$avg_NO",
                    "NO2": "$avg_NO2",
                    "O3": "$avg_O3",
                    "SO2": "$avg_SO2",
                    "PM2_5": "$avg_PM2_5",
                    "PM10": "$avg_PM10",
                    "NH3": "$avg_NH3"
                }
            }
        }
    }
]
results = list(db.WeatherDB.aggregate(pipeline))

data = []
for result in results:
    city = result["_id"]
    max_temps = result["max_temperature_days"]
    min_temps = result["min_temperature_days"]
    avg_rainfall = result["average_rainfall_per_day"]
    avg_pollution = result["avg_pollution"]

    data.append({
        "city": city,
        "max_temperature_days": max_temps,
        "min_temperature_days": min_temps,
        "average_rainfall_per_day": avg_rainfall,
        "avg_pollution": avg_pollution
    })

df = pd.DataFrame(data)
df.set_index("city", inplace = True)
print(df)


                                              max_temperature_days  \
city                                                                 
Essex            [12.600000000000023, 17.230000000000018, 8.900...   
Kent             [11.730000000000018, 4.25, 28.900000000000034,...   
Surrey           [22.54000000000002, 11.129999999999995, 14.730...   
West Sussex      [12.430000000000007, 18.54000000000002, 14.650...   
East Sussex      [11.689999999999998, 15.230000000000018, 14.45...   
London           [19.450000000000045, 12.860000000000014, 10.15...   
Hertfordshire    [21.29000000000002, 18.52000000000004, 14.6899...   
Buckinghamshire  [13.150000000000034, 22.970000000000027, 13.12...   

                                              min_temperature_days  \
city                                                                 
Essex            [3.140000000000043, 9.340000000000032, 3.53000...   
Kent             [2.7900000000000205, -3.269999999999982, 15.02...   
Surrey           [1

### Function to calculate suitability score of crop for location based on temperature, rain and pollution.
- `temperature` score: number of days in a year outside the temperature range of crop/number of days in the year
- `rainfall score`: 1-percentage of average rainfall greater/smaller than maximum/minimum rainfall
- `pollution score`: crop tolarance to pollutant are graded low,middle and high. quantify low, middle, high with the data average and standard deviation. minus mark when amount of pollutant exceeds the quantified tolarance level. 

In [9]:
def calculate_suitability_score(city_data, crop_data):
    """
    Calculate suitability score for a crop based on temperature, rainfall, and pollution.
    
    Args:
        city_data (pd.Series): Row of weather data for a city.
        crop_data (pd.Series): Row of crop data.
    
    Returns:
        float: Suitability score for the crop in the city.
    """
#temperature score
    temp_min_crop = crop_data["TempMinC"]
    temp_max_crop = crop_data["TempMaxC"]
    max_temps = city_data["max_temperature_days"]
    min_temps = city_data["min_temperature_days"]
    
    days_outside_temp = sum(
        (t > temp_max_crop or t < temp_min_crop) for t in max_temps + min_temps
    )
    temp_score = 100 * (1 - days_outside_temp / (len(max_temps)))  
     
     
#rainfall score     
    rain_min_crop = crop_data["RainMinMM"]
    rain_max_crop = crop_data["RainMaxMM"]
    avg_rainfall = city_data["average_rainfall_per_day"]*crop_data['GrowthCycleDays']
    if rain_min_crop <=  avg_rainfall <=  rain_max_crop:
        rain_score = 100
    elif rain_max_crop<avg_rainfall:
        rain_score = max(0,100*(avg_rainfall-rain_max_crop)/rain_max_crop)
    else:
        rain_score = max(0,100*(rain_min_crop-avg_rainfall)/avg_rainfall)


#pollution score
    pollution_factors = ["CO", "NO", "NO2", "O3", "SO2", "PM2_5", "PM10", "NH3"]
    avg_pollution_dict = city_data["avg_pollution"][0]  
    pollution_values = {factor: [] for factor in pollution_factors}
    for pollution in city_data["avg_pollution"]:  
        for factor in pollution_factors:
            value = pollution.get(factor, None)
            if value is not None:  
                pollution_values[factor].append(value)
    pollution_ranges = {}
    for factor, values in pollution_values.items():
        if not values: 
            continue

        avg_value = sum(values) / len(values)  
        std_dev = (sum((x - avg_value) ** 2 for x in values) / len(values)) ** 0.5  # Standard deviation
        low_threshold = avg_value - 0.68 * std_dev   #lower than 75% under normal distribution
        high_threshold = avg_value + 0.68 * std_dev   #larger than 75% under normal distribution
        pollution_ranges[factor] = {
            "Low": low_threshold,
            "Medium": avg_value,
            "High": high_threshold,
        }


    pollution_score = 0
    for factor in pollution_factors:
        city_value = avg_pollution_dict.get(factor, 0)
        crop_requirement = crop_data[f"AirPoll_{factor}"]
        # Determine the crop's ideal range based on its requirement
        ranges = pollution_ranges.get(factor, None)  # Handle missing ranges
        if ranges is None:
            continue  # Skip this factor if no valid range is available
        if crop_requirement ==  "Low" and city_value <=  ranges["Low"]:
            score = 1  
        elif crop_requirement ==  "Medium" and city_value <=  ranges["High"]:
            score = 1  
        elif crop_requirement ==  "High":
            score = 1  
        else:
            score = max(0, 1 - city_value - ranges[f"{crop_requirement}"] / ranges[f"{crop_requirement}"])
        pollution_score +=  score
    pollution_score = (pollution_score / len(pollution_factors)) * 100

    return 0.5 * temp_score + 0.3 * rain_score + 0.2 * pollution_score

## Load data from collection crops

In [10]:
db = client['WeatherDB']
crops_info = list(db['crops'].find())
crops_df = pd.DataFrame(crops_info)


## Apply give suitability score function to every city and crop and give a top 10 rank

In [11]:
def rank_crops_for_cities(df, crops_df):
    """
    Generate top 10 suitable crops for each city.

    Args:
        df (pd.DataFrame): DataFrame with city weather data, where the index is the city name.
        crops_df (pd.DataFrame): DataFrame with crop information.

    Returns:
        dict: Dictionary with cities as keys and top 10 crops as values.
    """
    city_rankings = {}

    # Iterate over each city (from the index of df)
    for city in df.index.unique():
        city_data = df.loc[city]  # Get city-specific data from the index
        scores = []

        # Iterate over each crop and calculate its suitability score for the city
        for _, crop in crops_df.iterrows():
            score = calculate_suitability_score(city_data, crop)  # Your scoring function
            scores.append((crop["CropName"], score))

        # Sort crops by their scores in descending order
        scores.sort(key = lambda x: x[1], reverse = True)

        # Take the top 10 crops for the city
        top_crops = [crop for crop, _ in scores[:10]]
        city_rankings[city] = top_crops

    return city_rankings

city_crop_rankings = rank_crops_for_cities(df, crops_df)

print(city_crop_rankings)

{'Essex': ['Pears', 'Xalora', 'Wheat', 'Rye', 'Plums', 'Barley', 'Triticale', 'Apples', 'Alveria', 'Halinia'], 'Buckinghamshire': ['Xalora', 'Wheat', 'Plums', 'Pears', 'Apples', 'Rye', 'Alveria', 'Triticale', 'Radish', 'Walnuts'], 'London': ['Xalora', 'Wheat', 'Plums', 'Apples', 'Alveria', 'Pears', 'Triticale', 'Barley', 'Tarsia', 'Elena'], 'Hertfordshire': ['Xalora', 'Wheat', 'Plums', 'Pears', 'Triticale', 'Alveria', 'Barley', 'Sunflowers', 'Halinia', 'Clarinda'], 'West Sussex': ['Plums', 'Xalora', 'Wheat', 'Rye', 'Pears', 'Triticale', 'Apples', 'Alveria', 'Barley', 'Elena'], 'East Sussex': ['Plums', 'Triticale', 'Xalora', 'Wheat', 'Pears', 'Alveria', 'Barley', 'Clarinda', 'Sunflowers', 'Elena'], 'Surrey': ['Radish', 'Wheat', 'Mint', 'Plums', 'Apples', 'Rye', 'Walnuts', 'Hazelnuts', 'Halinia', 'Rapeseed (Canola)'], 'Kent': ['Plums', 'Triticale', 'Xalora', 'Wheat', 'Rye', 'Pears', 'Apples', 'Alveria', 'Barley', 'Elena']}
